# Gemma3 270M: KerasHub export and Tunix SFT

Integration example for [keras-hub #3236](https://github.com/keras-team/keras-hub/issues/3236).

Load the pretrained **Gemma3 270M IT** checkpoint in KerasHub, export it to safetensors, load it in Tunix, compare logits and parameter counts, run two SFT steps, and save/restore trained parameters.

The saved output was executed locally on CPU with Python 3.12. Rerunning requires authorized Hugging Face access to `google/gemma-3-270m-it`. No authentication tokens are included.

In [1]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

COMMIT = "8a16f47f5ce38e86feb8cc6cdc537c54053fbee7"
assert sys.version_info >= (3, 12), "Use a Python 3.12 runtime."
local_repo = os.environ.get("TUNIX_REVIEW_REPO")
if local_repo:
    repo = Path(local_repo).resolve()
else:
    repo = Path(tempfile.mkdtemp()) / "keras-hub"
    subprocess.run(
        [
            "git",
            "clone",
            "-q",
            "https://github.com/bzantium/keras-hub.git",
            str(repo),
        ],
        check=True,
    )
    subprocess.run(["git", "checkout", "-q", COMMIT], cwd=repo, check=True)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-e",
            str(repo),
            "-r",
            str(repo / "tools/tunix/requirements.txt"),
        ],
        check=True,
    )
head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repo, text=True
).strip()
assert head == COMMIT, head
print("KerasHub commit:", head)
print("Python:", sys.version.split()[0])
os.environ["KERAS_BACKEND"] = "jax"
os.environ["JAX_PLATFORMS"] = "cpu"
sys.path.insert(0, str(repo))

KerasHub commit: 8a16f47f5ce38e86feb8cc6cdc537c54053fbee7
Python: 3.12.13


## Pretrained checkpoint verification and training

Accept the Gemma model terms and authenticate with an authorized Hugging Face account before running this cell. Keep tokens out of the notebook.

The checkpoint tokenizer encodes the instruction **"Say hello."** and completion **"Hello!"**. Only completion and turn-ending tokens contribute to loss. Training is two full-fine-tuning steps with SGD at `1e-5`. The loss is measured on that same example; a reduction shows the update path works, not generalization.

Before training, all logits are compared in float32 with `atol=1e-3, rtol=1e-4`, and argmax tokens must agree at every input position. The measured maximum/mean errors are reported below. A separate HF float32 model loaded from the same export gave maximum errors of `8.24e-4` for KerasHub and `6.52e-4` for Tunix, also with identical argmax tokens. These results cover this input and CPU environment.

The script also asserts equal parameter counts, changed parameters, decreased same-batch loss, and identical logits after restoring the trained Orbax parameters. Allow several GB of memory for model copies and gradients.

In [2]:
import json

env = dict(
    os.environ, KERAS_BACKEND="jax", JAX_PLATFORMS="cpu", PYTHONPATH=str(repo)
)
pretrained_dir = Path(tempfile.mkdtemp(prefix="gemma3-270m-tunix-"))
result = subprocess.run(
    [
        sys.executable,
        str(repo / "tools/tunix/finetune_gemma3.py"),
        "--output_dir",
        str(pretrained_dir),
    ],
    cwd=repo,
    env=env,
    capture_output=True,
    text=True,
)
if result.returncode:
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()
print(result.stdout)
pretrained_results = json.loads((pretrained_dir / "results.json").read_text())
assert pretrained_results["trained_checkpoint_logits_match"]
print("Real Gemma3 270M IT integration checks passed.")

Logits: max abs diff=0.000328064, mean abs diff=4.96033e-05
{
  "preset": "hf://google/gemma-3-270m-it",
  "versions": {
    "keras": "3.15.1",
    "google-tunix": "0.1.7",
    "jax": "0.11.0",
    "flax": "0.12.8",
    "safetensors": "0.7.0"
  },
  "devices": [
    "cpu:0"
  ],
  "parameters": 268098176,
  "max_absolute_logit_error": 0.00032806396484375,
  "mean_absolute_logit_error": 4.960330988978967e-05,
  "logit_atol": 0.001,
  "logit_rtol": 0.0001,
  "argmax_agreement": 1.0,
  "training_steps": 2,
  "learning_rate": 1e-05,
  "initial_loss": 4.572533130645752,
  "final_loss": 0.014963150024414062,
  "max_logit_change_after_training": 36.956729888916016,
  "max_parameter_change": 0.0005773517768830061,
  "trained_checkpoint_logits_match": true
}

Real Gemma3 270M IT integration checks passed.


## Reject empty target masks

An all-false mask and a mask active only at the first token have no next-token targets. Both must raise an error before evaluating the model. A single active next-token target must still produce the expected cross-entropy.

In [3]:
from unittest import mock

import jax.numpy as jnp
import numpy as np
from tunix.sft.peft_trainer import TrainingInput

from tools.tunix import finetune_gemma3 as example

tokens = np.array([[2, 3, 4, 5, 6, 7, 8, 9]], dtype="int32")
for mask in (
    np.zeros_like(tokens, dtype=bool),
    np.array([[True] + [False] * 7]),
):
    batch = TrainingInput(input_tokens=tokens, input_mask=mask)
    with mock.patch.object(example, "logits") as forward:
        try:
            example.loss(None, batch)
        except ValueError as error:
            assert "at least one target token" in str(error)
        else:
            raise AssertionError("Empty targets were accepted.")
        forward.assert_not_called()

batch = TrainingInput(
    input_tokens=tokens, input_mask=np.array([[False] * 7 + [True]])
)
with mock.patch.object(example, "logits", return_value=jnp.zeros((1, 8, 64))):
    np.testing.assert_allclose(example.loss(None, batch), np.log(64), rtol=1e-6)
print("Empty and first-token-only masks rejected; single-target loss verified.")

/tmp/keras-hub-tunix-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


I0000 00:00:1794143290.635867  394763 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794143290.637054  394763 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1794143292.000945  394763 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794143292.001398  394763 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Empty and first-token-only masks rejected; single-target loss verified.


## Scope

Gemma4, LoRA, distributed training, padded batches, general training quality evaluation, and importing Tunix-trained weights into KerasHub remain outside this example. The checks above use the actual KerasHub and Tunix implementations without modifying either model's arithmetic.